# 1-레이어 어텐션: QK 회로와 OV 회로 - QK 회로: 어텐션 패턴 결정

- Tutorial ID: `tut-2-1`
- Tutorial: 1-레이어 어텐션: QK 회로와 OV 회로
- Section ID: `s2-1-1`
- Section: QK 회로: 어텐션 패턴 결정


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

print("=" * 55)
print("QK 회로 (Query-Key Circuit) 분석")
print("=" * 55)

# 하이퍼파라미터
vocab_size = 6
d_model = 4
d_head = 2

vocab = ['A', 'B', 'C', 'D', 'E', 'F']

np.random.seed(0)

# 가중치 초기화
W_E = np.random.randn(vocab_size, d_model) * 0.5   # 임베딩
W_Q = np.random.randn(d_model, d_head) * 0.3       # 쿼리
W_K = np.random.randn(d_model, d_head) * 0.3       # 키

def softmax(x, axis=-1):
        x = x - np.max(x, axis=axis, keepdims=True)
    return np.exp(x) / np.sum(np.exp(x), axis=axis, keepdims=True)

In [ ]:
# QK 회로 행렬 계산

In [ ]:
# W_QK = W_Q @ W_K^T: (d_model, d_model)
W_QK = W_Q @ W_K.T
print(f"W_QK = W_Q @ W_K^T shape: {W_QK.shape}")

# 확장된 QK 회로 (직접 어휘 → 어텐션 점수 매핑)
# C_QK = W_E^T @ W_QK @ W_E: (vocab, vocab)
C_QK = W_E @ W_QK @ W_E.T
print(f"C_QK = W_E^T @ W_QK @ W_E shape: {C_QK.shape}")

print("
확장된 QK 행렬 (행=쿼리 토큰, 열=키 토큰):")
print("     " + "  ".join(f"{v:4s}" for v in vocab))
for i, vi in enumerate(vocab):
        scores = " ".join(f"{C_QK[i,j]:4.1f}" for j in range(vocab_size))
    print(f"  {vi}: {scores}")

In [ ]:
# 실제 어텐션 계산 과정

In [ ]:
print("
" + "=" * 55)
print("실제 어텐션 계산 과정 (시퀀스: A B C D)")
print("=" * 55)

# 시퀀스 처리
sequence = [0, 1, 2, 3]  # A, B, C, D
seq_len = len(sequence)

# 임베딩 조회
X = W_E[sequence]  # (seq_len, d_model)
print(f"시퀀스 임베딩 X.shape: {X.shape}")

# 쿼리, 키 계산
Q = X @ W_Q  # (seq_len, d_head)
K = X @ W_K  # (seq_len, d_head)
print(f"Q.shape: {Q.shape}, K.shape: {K.shape}")

# 스케일된 어텐션 점수
scale = np.sqrt(d_head)
attn_scores = Q @ K.T / scale  # (seq_len, seq_len)

# 인과적 마스크 (causal mask)
mask = np.triu(np.full((seq_len, seq_len), -1e9), k=1)
attn_scores_masked = attn_scores + mask

# 어텐션 가중치
attn_weights = softmax(attn_scores_masked, axis=-1)

print("
어텐션 가중치 행렬 (행=쿼리 위치, 열=키 위치):")
print("    A    B    C    D")
for i, tok_id in enumerate(sequence):
        weights = " ".join(f"{w:.3f}" for w in attn_weights[i])
    print(f"  {vocab[tok_id]}: {weights}")

print("
해석:")
print("  - 각 행의 합 = 1.0")
print("  - 대각선 아래: 이전 토큰 참조 가능")
print("  - 대각선 위: 마스킹됨 (미래 참조 불가)")
print()

# 어텐션 패턴 시각화
print("시각화 (밝을수록 높은 어텐션):")
print("    " + " ".join(vocab[:seq_len]))
for i in range(seq_len):
    row = ""
        for j in range(seq_len):
        w = attn_weights[i, j]
        if w > 0.6:
            row += "█ "
        elif w > 0.3:
            row += "▓ "
        elif w > 0.1:
            row += "░ "
        else:
            row += "· "
    print(f"  {vocab[sequence[i]]}: {row}")